# Day 30 - LRU and LFU caches: choosing what to throw away

A cache is two things bolted together: a bounded map, and a *policy* for picking
a victim when the map is full.  The map half is Day 09's hash table and holds no
surprises.  Everything that makes a cache fast or slow lives in the policy.

This notebook builds the two policies interviewers ask for - LRU (LeetCode 146)
and LFU (LeetCode 460) - measures them against the provably optimal offline
policy, shows the two ways each of them fails, and ends at the shape a real
inference server ships: one heap plus a key function.

In [1]:
import heapq
import random
import time
from collections import OrderedDict, defaultdict

## 1. The ceiling nobody can reach

Belady's MIN is the optimal policy: evict the item whose *next* use is furthest
in the future.  It needs to see the future, so it cannot be implemented online -
but it tells us how much room a real policy still has.

In [2]:
def belady_hits(trace, capacity):
    """Evict the item whose NEXT use is furthest away (or never).

    Provably optimal, and impossible online: it needs the future.  We use it
    only as a yardstick - no online policy can beat this number.
    """
    # next_use[i] = the next index > i where the same key appears
    nxt = [len(trace)] * len(trace)
    last_seen = {}
    for i in range(len(trace) - 1, -1, -1):
        nxt[i] = last_seen.get(trace[i], len(trace))
        last_seen[trace[i]] = i

    cache = {}                       # key -> next use index
    hits = 0
    for i, key in enumerate(trace):
        if key in cache:
            hits += 1
        elif len(cache) < capacity:
            pass
        else:
            victim = max(cache, key=lambda k: cache[k])
            del cache[victim]
        cache[key] = nxt[i]
    return hits

trace = [1, 2, 3, 1, 4, 1, 2, 5, 1, 2, 3, 4, 5]
print('optimal hits on a 13-request trace, capacity 3:',
      belady_hits(trace, 3), '/', len(trace))

optimal hits on a 13-request trace, capacity 3: 6 / 13


## 2. LRU: a hash map for lookup, a linked list for order

`get` has to be O(1) *and* has to change the order, which is exactly what a
doubly linked list gives you: the map finds the node, the node splices itself
out and back in at the front.  Two sentinels remove every "am I at the end?"
branch.

In [3]:
class Node:
    """Doubly linked list node.  __slots__ because there is one per entry."""
    __slots__ = ('key', 'val', 'prev', 'next')

    def __init__(self, key=None, val=None):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    """LeetCode 146.  Hash map for lookup, doubly linked list for order.

    The list is kept most-recent-first between two sentinels, so no branch
    anywhere has to ask "am I at the end?".
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.map = {}
        self.head = Node()            # sentinel: most recent side
        self.tail = Node()            # sentinel: least recent side
        self.head.next = self.tail
        self.tail.prev = self.head
        self.hits = self.misses = self.evictions = 0

    # --- list surgery, both O(1) ---
    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _push_front(self, node):
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        node = self.map.get(key)
        if node is None:
            self.misses += 1
            return -1
        self.hits += 1
        self._unlink(node)            # a READ is a use: refresh the order
        self._push_front(node)
        return node.val

    def put(self, key, value):
        node = self.map.get(key)
        if node is not None:
            node.val = value
            self._unlink(node)
            self._push_front(node)
            return
        if len(self.map) >= self.capacity:
            victim = self.tail.prev   # the sentinel makes this one line
            self._unlink(victim)
            del self.map[victim.key]
            self.evictions += 1
        node = Node(key, value)
        self.map[key] = node
        self._push_front(node)

    def order(self):
        """Most recent first - for the diagrams and the tests."""
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.key)
            cur = cur.next
        return out

In [4]:
c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
print('after put(1,1), put(2,2):', c.order())
print('get(1) ->', c.get(1), '  order now:', c.order())
c.put(3, 3)
print('put(3,3) evicted 2, not 1 ->', c.order())
print('get(2) ->', c.get(2))

after put(1,1), put(2,2): [2, 1]
get(1) -> 1   order now: [1, 2]
put(3,3) evicted 2, not 1 -> [3, 1]
get(2) -> -1


A read is a use.  That single line in `get` - moving the node to the front - is
the entire difference between LRU and FIFO, and forgetting it raises nothing.

In [5]:
class FIFOCache:
    """LRU with the refresh on `get` deleted - which is exactly FIFO.

    This is the single most common LRU bug: the eviction order stops tracking
    use and starts tracking arrival.  Nothing crashes; the hit rate just drops.
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.d = OrderedDict()
        self.hits = self.misses = self.evictions = 0

    def get(self, key):
        if key not in self.d:
            self.misses += 1
            return -1
        self.hits += 1
        return self.d[key]            # <- no move_to_end

    def put(self, key, value):
        if key not in self.d and len(self.d) >= self.capacity:
            self.d.popitem(last=False)
            self.evictions += 1
        self.d[key] = value

    def order(self):
        return list(reversed(self.d))

f = FIFOCache(2)
f.put(1, 1); f.put(2, 2); f.get(1); f.put(3, 3)
print('FIFO after the same script:', f.order())
print('get(2) ->', f.get(2), '  (LeetCode 146 wants -1 here)')

FIFO after the same script: [3, 2]
get(2) -> 2   (LeetCode 146 wants -1 here)


## 3. `OrderedDict` is that linked list

CPython's `OrderedDict` is a hash map plus a doubly linked list, with
`move_to_end` and `popitem(last=False)` already written in C.  Same policy,
a third of the code.

In [6]:
class LRUCacheOrderedDict:
    """The same policy in six lines, because OrderedDict IS that linked list."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.d = OrderedDict()
        self.hits = self.misses = self.evictions = 0

    def get(self, key):
        if key not in self.d:
            self.misses += 1
            return -1
        self.hits += 1
        self.d.move_to_end(key)
        return self.d[key]

    def put(self, key, value):
        if key in self.d:
            self.d.move_to_end(key)
        elif len(self.d) >= self.capacity:
            self.d.popitem(last=False)
            self.evictions += 1
        self.d[key] = value

    def order(self):
        return list(reversed(self.d))

a, b = LRUCache(3), LRUCacheOrderedDict(3)
for k in (1, 2, 3, 1, 4, 2, 5, 1):
    if a.get(k) == -1: a.put(k, k)
    if b.get(k) == -1: b.put(k, k)
print('hand-rolled:', a.order())
print('OrderedDict:', b.order())
assert a.order() == b.order()

hand-rolled: [1, 5, 2]
OrderedDict: [1, 5, 2]


The version people write first uses a plain list and `remove` / `insert(0)`.
It makes exactly the same eviction decisions and is O(n) per operation.

In [7]:
class LRUCacheList:
    """The version people write first: a plain list, move-to-front on use.

    Same eviction decisions as LRUCache, but every hit scans the list, so it
    is O(n) per operation instead of O(1).  Correct, and quadratic.
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.order_ = []              # most recent first
        self.val = {}
        self.hits = self.misses = self.evictions = 0
        self.scanned = 0              # cells touched by list.remove / index

    def get(self, key):
        if key not in self.val:
            self.misses += 1
            return -1
        self.hits += 1
        self.scanned += self.order_.index(key) + 1
        self.order_.remove(key)
        self.order_.insert(0, key)
        return self.val[key]

    def put(self, key, value):
        if key in self.val:
            self.scanned += self.order_.index(key) + 1
            self.order_.remove(key)
        elif len(self.val) >= self.capacity:
            victim = self.order_.pop()
            del self.val[victim]
            self.evictions += 1
        self.order_.insert(0, key)
        self.val[key] = value

    def order(self):
        return list(self.order_)

t = {}
rng = random.Random(7)
probe = [rng.randrange(4000) for _ in range(40000)]
for name, cache in (('linked list O(1)', LRUCache(2000)),
                    ('python list O(n)', LRUCacheList(2000))):
    t0 = time.perf_counter()
    for k in probe:
        if cache.get(k) == -1:
            cache.put(k, k)
    t[name] = time.perf_counter() - t0
    print('%-18s %6.1f ms' % (name, 1000 * t[name]))
print('ratio: %.0fx' % (t['python list O(n)'] / t['linked list O(1)']))

linked list O(1)     75.2 ms
python list O(n)    668.1 ms
ratio: 9x


## 4. LFU in O(1): one bucket per use count

Counting uses is easy; evicting the minimum in O(1) is the trick.  Keep one
`OrderedDict` per frequency and a single `min_freq`.  A hit moves a key from
bucket `f` to bucket `f+1`; if that empties `f` and `f` was the minimum, the new
minimum is `f+1`.  An insert sets it to 1.  There is no third case, so
`min_freq` never searches.

In [8]:
class LFUCache:
    """Count uses, evict the least used, break ties by LRU.

    The trick that makes it O(1): one bucket per frequency, each bucket an
    ordered dict.  `min_freq` is the only global, and it can only ever move
    up by one (on a hit) or reset to 1 (on an insert), so it never searches.
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.val = {}                          # key -> value
        self.freq = {}                         # key -> use count
        self.buckets = defaultdict(OrderedDict)  # count -> keys, LRU first
        self.min_freq = 0
        self.hits = self.misses = self.evictions = 0

    def _touch(self, key):
        f = self.freq[key]
        del self.buckets[f][key]
        if not self.buckets[f]:
            del self.buckets[f]
            if self.min_freq == f:
                self.min_freq = f + 1          # the ONLY way min_freq grows
        self.freq[key] = f + 1
        self.buckets[f + 1][key] = None

    def get(self, key):
        if key not in self.val:
            self.misses += 1
            return -1
        self.hits += 1
        self._touch(key)
        return self.val[key]

    def put(self, key, value):
        if self.capacity <= 0:
            return
        if key in self.val:
            self.val[key] = value
            self._touch(key)
            return
        if len(self.val) >= self.capacity:
            victim, _ = self.buckets[self.min_freq].popitem(last=False)
            if not self.buckets[self.min_freq]:
                del self.buckets[self.min_freq]
            del self.val[victim], self.freq[victim]
            self.evictions += 1
        self.val[key] = value
        self.freq[key] = 1
        self.buckets[1][key] = None
        self.min_freq = 1                      # a new key resets the floor

    def snapshot(self):
        """{frequency: [keys, least recent first]} - for the diagrams."""
        return {f: list(keys) for f, keys in sorted(self.buckets.items())}

In [9]:
c = LFUCache(2)
c.put(1, 1); c.put(2, 2)
print('get(1) ->', c.get(1))
c.put(3, 3)                      # evicts 2: it has the lower count
print('get(2) ->', c.get(2))
print('get(3) ->', c.get(3))
c.put(4, 4)                      # 1 and 3 tie at 2 -> evict the older use
print('get(1) ->', c.get(1), ' get(3) ->', c.get(3), ' get(4) ->', c.get(4))

f = LFUCache(6)
for k in ('d', 'e', 'c', 'a', 'b'):
    f.put(k, k)
for k in ('a', 'a', 'b', 'b', 'c'):
    f.get(k)
print('buckets:', f.snapshot(), ' min_freq =', f.min_freq)

get(1) -> 1
get(2) -> -1
get(3) -> 3
get(1) -> -1  get(3) -> 3  get(4) -> 4
buckets: {1: ['d', 'e'], 2: ['c'], 3: ['a', 'b']}  min_freq = 1


The tie-break *is* the problem.  "Least frequently used" is ambiguous the moment
two keys share the lowest count, and LeetCode 460 checks that you break the tie
by recency.  Dropping the within-bucket order still passes most cases.

In [10]:
class LFUNoTieBreak:
    """LFU that keeps only counts and drops the within-bucket order.

    It still evicts a minimum-frequency key, so it is "an LFU" - but which
    one it picks is arbitrary, and LeetCode 460 checks exactly that.
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.val, self.freq = {}, {}
        self.hits = self.misses = self.evictions = 0

    def get(self, key):
        if key not in self.val:
            self.misses += 1
            return -1
        self.hits += 1
        self.freq[key] += 1
        return self.val[key]

    def put(self, key, value):
        if key in self.val:
            self.val[key] = value
            self.freq[key] += 1
            return
        if len(self.val) >= self.capacity:
            lo = min(self.freq.values())
            # "any key with the minimum count" - here the first inserted one
            victim = next(k for k in self.val if self.freq[k] == lo)
            del self.val[victim], self.freq[victim]
            self.evictions += 1
        self.val[key] = value
        self.freq[key] = 1

def script(cache):
    cache.put(1, 1); cache.put(2, 2)
    cache.get(2); cache.get(1)   # both at count 2, key 2 is the older use
    cache.put(3, 3)
    return cache.get(1), cache.get(2)

print('LRU tie-break   :', script(LFUCache(2)), '<- LeetCode 460')
print('insertion order :', script(LFUNoTieBreak(2)))

LRU tie-break   : (1, -1) <- LeetCode 460
insertion order : (-1, 2)


## 5. The shape a real server ships

sglang's radix cache does not implement five caches.  Each tree node carries
`last_access_time`, `hit_count` and `creation_time`, and a strategy object turns
a node into a sort key; eviction is one heap over the candidate leaves, and the
policy is a command-line flag (`--radix-eviction-policy`).

In [11]:
class Entry:
    __slots__ = ('key', 'val', 'last_access', 'hit_count', 'created')

    def __init__(self, key, val, clock):
        self.key, self.val = key, val
        self.last_access = self.created = clock
        self.hit_count = 0

def priority_lru(e):
    return e.last_access

def priority_lfu(e):
    return (e.hit_count, e.last_access)

def priority_fifo(e):
    return e.created

def priority_mru(e):
    return -e.last_access

def make_priority_slru(threshold=2):
    """Segmented LRU: one bit of protection on top of LRU.

    Entries that have been hit `threshold` times move to the protected
    segment and are evicted only after every probationary entry is gone.
    That single bit is what stops a scan from wiping the cache.
    """
    def priority(e):
        protected = 1 if e.hit_count >= threshold else 0
        return (protected, e.last_access)
    return priority

POLICIES = {'lru': priority_lru, 'lfu': priority_lfu, 'fifo': priority_fifo,
            'mru': priority_mru, 'slru': make_priority_slru(2)}


class PolicyCache:
    """A cache whose eviction rule is a function from entry to sort key."""

    def __init__(self, capacity, priority):
        self.capacity = capacity
        self.priority = priority
        self.map = {}
        self.clock = 0
        self.hits = self.misses = self.evictions = 0

    def _tick(self):
        self.clock += 1
        return self.clock

    def get(self, key):
        e = self.map.get(key)
        if e is None:
            self.misses += 1
            return -1
        self.hits += 1
        e.hit_count += 1
        e.last_access = self._tick()
        return e.val

    def put(self, key, value):
        e = self.map.get(key)
        if e is not None:
            e.val = value
            e.hit_count += 1
            e.last_access = self._tick()
            return
        if len(self.map) >= self.capacity:
            # the same heapify-the-candidates shape sglang uses
            victim = heapq.nsmallest(
                1, self.map.values(), key=self.priority)[0]
            del self.map[victim.key]
            self.evictions += 1
        self.map[key] = Entry(key, value, self._tick())

## 6. The policies only differ when the workload does

Four traces: skewed popularity, a cyclic scan, a hot set that keeps moving, and
a realistic mix of a hot set with cold scans cutting through it.

In [12]:
def zipf_trace(n_keys=200, length=4000, alpha=1.1, seed=0):
    """Skewed popularity: a few keys get most of the requests."""
    rng = random.Random(seed)
    weights = [1.0 / (i ** alpha) for i in range(1, n_keys + 1)]
    total = sum(weights)
    cum, acc = [], 0.0
    for w in weights:
        acc += w / total
        cum.append(acc)
    out = []
    for _ in range(length):
        x = rng.random()
        lo, hi = 0, n_keys - 1
        while lo < hi:                      # Day 23, still earning its keep
            mid = (lo + hi) // 2
            if cum[mid] < x:
                lo = mid + 1
            else:
                hi = mid
        out.append(lo)
    return out

def scan_trace(n_keys=33, rounds=120):
    """A loop over one more key than fits.  The classic LRU killer."""
    return [i % n_keys for i in range(n_keys * rounds)]

def shifting_trace(n_hot=16, hot_len=300, phases=12, n_keys=400, seed=1):
    """A hot set that moves.  Yesterday's stars are today's dead weight."""
    rng = random.Random(seed)
    out = []
    for p in range(phases):
        hot = [rng.randrange(n_keys) for _ in range(n_hot)]
        for _ in range(hot_len):
            out.append(rng.choice(hot))
    return out

def mixed_trace(n_hot=24, n_cold=400, length=6000, scan_every=40, seed=2):
    """A hot working set, interrupted by a long scan every so often."""
    rng = random.Random(seed)
    out, cold = [], 0
    for i in range(length):
        if i % scan_every == 0:
            for _ in range(12):             # a burst of never-reused keys
                out.append(1000 + cold)
                cold = (cold + 1) % n_cold
        out.append(rng.randrange(n_hot))
    return out

def run_trace(cache, trace):
    """A read-through cache: miss -> fetch -> insert."""
    for key in trace:
        if cache.get(key) == -1:
            cache.put(key, key)
    return cache.hits / len(trace)

def compare_policies(trace, capacity):
    """Hit rate of every policy, plus the offline optimum."""
    rows = {}
    for name, prio in POLICIES.items():
        rows[name] = run_trace(PolicyCache(capacity, prio), trace)
    rows['opt'] = belady_hits(trace, capacity) / len(trace)
    return rows

WORKLOADS = {'zipf': (zipf_trace(), 32),
             'scan': (scan_trace(), 32),
             'shifting hot set': (shifting_trace(), 32),
             'hot set + cold scans': (mixed_trace(), 32)}

names = ['opt'] + sorted(POLICIES)
print('%-22s %s' % ('workload', ' '.join('%6s' % n for n in names)))
for label, (tr, cap) in WORKLOADS.items():
    rows = compare_policies(tr, cap)
    print('%-22s %s' % (label, ' '.join('%5.1f%%' % (100 * rows[n])
                                        for n in names)))

workload                  opt   fifo    lfu    lru    mru   slru
zipf                    79.0%  57.6%  71.6%  63.2%  22.2%  69.5%
scan                    96.1%   0.0%   0.0%   0.0%  96.1%   0.0%
shifting hot set        95.7%  94.8%  26.8%  94.9%  27.3%  30.5%
hot set + cold scans    77.0%  55.6%  76.6%  64.1%  16.7%  76.4%


## 7. Two failure modes, pointing in opposite directions

**Scan pollution.**  Loop over one more key than fits and LRU evicts exactly the
key it is about to ask for - every time, forever.  Note that LFU and SLRU do not
save you here: on a pure scan nothing is ever hit twice, so no counter rises and
no entry earns protection.  Only MRU, which throws away what it just used,
breaks the lockstep.

**LFU never forgets.**  When the hot set moves, yesterday's favourites keep
their counts and squat in the cache.  That is why real LFUs age their counters,
and why sglang's LFU key is `(hit_count, last_access_time)` rather than the
count alone.

In [13]:
loop = scan_trace(n_keys=33, rounds=120)
for n in ('lru', 'lfu', 'slru', 'mru'):
    print('%-5s on a 33-key loop through 32 slots: %5.1f%%'
          % (n, 100 * run_trace(PolicyCache(32, POLICIES[n]), loop)))

print()
moving = shifting_trace()
rows = compare_policies(moving, 32)
for n in ('opt', 'lru', 'lfu', 'slru'):
    print('%-5s on a hot set that keeps moving: %5.1f%%' % (n, 100 * rows[n]))

print()
mix = mixed_trace()
for n in ('lru', 'slru'):
    print('%-5s on a hot set plus cold scans: %5.1f%%'
          % (n, 100 * run_trace(PolicyCache(32, POLICIES[n]), mix)))

lru   on a 33-key loop through 32 slots:   0.0%
lfu   on a 33-key loop through 32 slots:   0.0%
slru  on a 33-key loop through 32 slots:   0.0%
mru   on a 33-key loop through 32 slots:  96.1%

opt   on a hot set that keeps moving:  95.7%
lru   on a hot set that keeps moving:  94.9%
lfu   on a hot set that keeps moving:  26.8%
slru  on a hot set that keeps moving:  30.5%

lru   on a hot set plus cold scans:  64.1%
slru  on a hot set plus cold scans:  76.4%


## 8. LeetCode 432 - the bucket trick on its own

All O(1) Data Structure is LFU's frequency buckets without the cache around
them: keys with equal counts share a bucket, the buckets form a sorted doubly
linked list, and an increment only ever moves a key to the neighbouring bucket.

In [14]:
class AllOne:
    """inc / dec / getMaxKey / getMinKey, all O(1).

    Same insight as LFU: keys with equal counts live together in a bucket,
    and the buckets themselves form a sorted doubly linked list, so a count
    only ever moves a key to the neighbouring bucket.
    """

    class Bucket:
        __slots__ = ('count', 'keys', 'prev', 'next')

        def __init__(self, count=0):
            self.count = count
            self.keys = set()
            self.prev = self.next = None

    def __init__(self):
        self.head = self.Bucket()              # sentinels, counts -inf / +inf
        self.tail = self.Bucket()
        self.head.next, self.tail.prev = self.tail, self.head
        self.where = {}                        # key -> bucket

    def _insert_after(self, node, count):
        b = self.Bucket(count)
        b.prev, b.next = node, node.next
        node.next.prev = b
        node.next = b
        return b

    def _drop(self, b):
        b.prev.next = b.next
        b.next.prev = b.prev

    def inc(self, key):
        if key not in self.where:
            first = self.head.next
            if first is self.tail or first.count != 1:
                first = self._insert_after(self.head, 1)
            first.keys.add(key)
            self.where[key] = first
            return
        cur = self.where[key]
        nxt = cur.next
        if nxt is self.tail or nxt.count != cur.count + 1:
            nxt = self._insert_after(cur, cur.count + 1)
        nxt.keys.add(key)
        self.where[key] = nxt
        cur.keys.discard(key)
        if not cur.keys:
            self._drop(cur)

    def dec(self, key):
        if key not in self.where:
            return
        cur = self.where[key]
        if cur.count == 1:
            del self.where[key]
        else:
            prv = cur.prev
            if prv is self.head or prv.count != cur.count - 1:
                prv = self._insert_after(cur.prev, cur.count - 1)
            prv.keys.add(key)
            self.where[key] = prv
        cur.keys.discard(key)
        if not cur.keys:
            self._drop(cur)

    def getMaxKey(self):
        return '' if self.tail.prev is self.head else next(iter(self.tail.prev.keys))

    def getMinKey(self):
        return '' if self.head.next is self.tail else next(iter(self.head.next.keys))

    def buckets(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append((cur.count, sorted(cur.keys)))
            cur = cur.next
        return out

o = AllOne()
for w in ('hello', 'hello', 'world', 'leet', 'leet', 'leet'):
    o.inc(w)
print('max:', o.getMaxKey(), ' min:', o.getMinKey(), ' buckets:', o.buckets())
o.dec('world'); o.dec('leet')
print('max:', o.getMaxKey(), ' min:', o.getMinKey(), ' buckets:', o.buckets())

max: leet  min: world  buckets: [(1, ['world']), (2, ['hello']), (3, ['leet'])]
max: hello  min: hello  buckets: [(2, ['hello', 'leet'])]


## 9. Complexity

| structure | get | put | eviction | space |
|---|---|---|---|---|
| LRU (map + linked list) | O(1) | O(1) | O(1) | O(capacity) |
| LRU (plain list) | O(n) | O(n) | O(1) | O(capacity) |
| LFU (frequency buckets) | O(1) | O(1) | O(1) | O(capacity) |
| policy heap (sglang shape) | O(1) | O(n) or O(log n) | O(n) scan / O(log n) heap | O(capacity) |
| Belady's MIN | - | - | needs the future | O(n) |

The heap version trades the constant-time eviction for the ability to swap
policies without touching the cache - a good trade when the candidate set is
small and the workload is the thing you are still tuning.

In [15]:
assert belady_hits([1, 2, 1, 2], 2) == 2
c = LRUCache(2); c.put(1, 1); c.put(2, 2); c.get(1); c.put(3, 3)
assert (c.get(2), c.get(1), c.get(3)) == (-1, 1, 3)
f = FIFOCache(2); f.put(1, 1); f.put(2, 2); f.get(1); f.put(3, 3)
assert f.get(2) == 2            # the bug, reproduced on purpose
l = LFUCache(2); l.put(1, 1); l.put(2, 2); l.get(1); l.put(3, 3)
assert l.get(2) == -1 and l.get(3) == 3
scan = scan_trace(n_keys=9, rounds=30)
assert run_trace(PolicyCache(8, POLICIES['lru']), scan) == 0.0
assert run_trace(PolicyCache(8, POLICIES['mru']), scan) > 0.5
z = zipf_trace(length=1500)
top = belady_hits(z, 16) / len(z)
for n in POLICIES:
    assert run_trace(PolicyCache(16, POLICIES[n]), z) <= top + 1e-12
o = AllOne()
for w in ('a', 'a', 'b'):
    o.inc(w)
assert (o.getMaxKey(), o.getMinKey()) == ('a', 'b')
print('all assertions passed')

all assertions passed
